# Experiment 1.2b-ii: Trajectory in Polar Coordinates -- (Q_t, P_t) Decomposition

## Scientific Motivation

Any matrix $W$ admits a **polar decomposition** $W = QP$, where $Q$ is orthogonal (lives on the Stiefel manifold $\mathrm{St}(n)$) and $P$ is symmetric positive semi-definite (lives in $\mathrm{Sym}^+$). This factorization cleanly separates:

- **Orientation** ($Q$): the rotational/reflective part of the linear map -- in the gauge-fixing interpretation, this is the "gauge" degree of freedom (equivalent networks related by orthogonal transformations).
- **Spectrum** ($P$): the stretching/scaling part -- this encodes the singular value structure, which directly determines the network's input-output map.

If Muon acts as a **renormalization-group gauge fixer**, its updates should predominantly rotate weights on the Stiefel manifold (changing $Q$) while leaving the spectrum ($P$) relatively unchanged. SGD, having no such geometric bias, should move freely in both directions.

## Hypothesis

> **H1 (Muon orientation dominance):** Under Muon optimization, the per-step change ratio $\|\\Delta Q\| / \|\\Delta W\| \approx 1$ and $\|\\Delta P\| / \|\\Delta W\| \approx 0$, meaning updates are predominantly orientation changes.
>
> **H2 (SGD mixed motion):** Under SGD, both $\|\\Delta Q\| / \|\\Delta W\|$ and $\|\\Delta P\| / \|\\Delta W\|$ are $O(1)$, meaning updates distribute across both orientation and spectrum.
>
> **H3 (Cumulative drift):** Over the full trajectory, Muon's cumulative orientation fraction $\|Q_t - Q_0\| / (\|Q_t - Q_0\| + \|P_t - P_0\|)$ should exceed SGD's.

## Critical Context from Experiment 1.2b-i

The Lyapunov exponent analysis (1.2b-i) revealed a **surprising** result: Muon is MORE chaotic in weight space but more stable in loss space. The benefit appears to be **directional**, not about stability. Neither optimizer clearly distinguished gauge from physical directions in Lyapunov terms. This experiment tests whether the polar decomposition provides a cleaner lens.

## Methodology

1. Train a 4-layer deep linear network ($32 \times 32$) with quadratic loss under both Muon and SGD+momentum.
2. At every step, compute the polar decomposition $W_t = Q_t P_t$ via SVD: $W = U\Sigma V^\top \Rightarrow Q = UV^\top$, $P = V\Sigma V^\top$.
3. Track per-step ratios ($\|\\Delta Q\|/\|\\Delta W\|$, $\|\\Delta P\|/\|\\Delta W\|$) and cumulative drift from initialization.
4. Compare the "orientation fraction" between optimizers across training.

## Expected Outcomes

- If H1-H3 hold: Muon's trajectory is confined to the Stiefel manifold, confirming the gauge-fixing interpretation.
- If the profiles are similar: the polar decomposition may not be the right geometric lens, and deeper fiber-bundle analysis may be needed.
- Intermediate results would suggest Muon has a partial geometric bias worth quantifying.

In [ ]:
"""
1.2b-ii: Trajectory in Polar Coordinates -- (Q_t, P_t) Decomposition
=====================================================================

At each step, decompose W_t = Q_t P_t (polar decomposition).
  Q_t lives on the Stiefel manifold (orthogonal matrices).
  P_t lives in Sym+ (symmetric positive semi-definite matrices).

Track how much of each weight update changes Q vs P:
  ||DeltaQ|| / ||DeltaW||   (orientation change fraction)
  ||DeltaP|| / ||DeltaW||   (spectrum change fraction)

Also track cumulative drift from initialization:
  ||Q_t - Q_0||_F   (total orientation drift)
  ||P_t - P_0||_F   (total spectrum drift)

HYPOTHESIS (to be tested, may be wrong given 1.2b-i results):
  Under Muon: ||DeltaQ||/||DeltaW|| ~ 1, ||DeltaP||/||DeltaW|| ~ 0
    (movement is predominantly in orientation, not spectrum)
  Under SGD:  both ratios are O(1)
    (movement in both orientation and spectrum)

CRITICAL CONTEXT from 1.2b-i:
  - Muon is MORE chaotic in weight space (higher Lyapunov) but more stable
    in loss space
  - The benefit is DIRECTIONAL, not stability
  - Neither optimizer distinguishes gauge from physical in Lyapunov terms
  - So this experiment may show something different from what was hypothesized

Setup: 4-layer deep linear net, 32x32, quadratic loss, 300 steps.
"""

In [ ]:
import numpy as np
import os

In [ ]:
np.random.seed(42)

## Configuration

**Parameter choices and their rationale:**

- **DIM = 32:** Large enough for non-trivial singular value structure (32 singular values per layer), small enough for exact SVD at every step without prohibitive cost.
- **NUM_LAYERS = 4:** Deep enough that gauge symmetries matter (inter-layer orthogonal transformations), shallow enough for stable gradient flow.
- **NUM_STEPS = 300:** Long enough to observe both transient and steady-state polar dynamics.
- **Initialization near identity ($I + 0.1 \cdot \text{noise}$):** Ensures $P_0 \approx I$ (all singular values near 1) and $Q_0 \approx I$, so that initial drift is small and changes are clearly attributable to optimization dynamics.
- **Newton-Schulz iterations (NS_ITERS=5):** Muon's core mechanism -- iteratively projects the gradient onto the orthogonal group. 5 iterations gives high-accuracy orthogonalization for 32x32 matrices.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 300
BATCH_SIZE = 64
LR_MUON = 0.005
MOMENTUM = 0.9
NS_ITERS = 5

print(f"Weight matrix dimensions: {DIM}x{DIM} ({DIM**2} parameters per layer)")
print(f"Total parameters across {NUM_LAYERS} layers: {NUM_LAYERS * DIM**2}")
print(f"Polar decomposition dimensionality: Q in O({DIM}) [{DIM*(DIM-1)//2} DOF], P in Sym+({DIM}) [{DIM*(DIM+1)//2} DOF]")
print(f"Gauge DOF per layer boundary: {DIM*(DIM-1)//2} (orthogonal group generators)")
print(f"Muon LR={LR_MUON}, SGD LR=auto-tuned, Momentum={MOMENTUM}")

In [ ]:
# Report steps
REPORT_STEPS = [50, 100, 200, 300]

In [ ]:
# Random target matrix (fixed)
W_target = np.random.randn(DIM, DIM) * 0.5

In [ ]:
# Random input data (fixed batch)
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3

print(f"Target matrix W_target: shape={W_target.shape}, Frobenius norm={np.linalg.norm(W_target, 'fro'):.4f}")
print(f"  Singular value range: [{np.min(np.linalg.svd(W_target, compute_uv=False)):.4f}, {np.max(np.linalg.svd(W_target, compute_uv=False)):.4f}]")
print(f"  Condition number: {np.linalg.cond(W_target):.2f}")
print(f"Input data X: shape={X_data.shape}, Frobenius norm={np.linalg.norm(X_data, 'fro'):.4f}")
print(f"  This is a FIXED batch -- no stochastic gradient noise, isolating optimizer geometry.")

In [ ]:
# Output directory
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

## Core Functions

### Deep Linear Network

The model is a **deep linear network**: $f(X) = W_L \cdot W_{L-1} \cdots W_1 \cdot X$. Despite being linear, this architecture has a rich optimization landscape with saddle points, gauge symmetries (any pair of adjacent layers can be transformed as $W_{i+1} \to W_{i+1} A^{-1}$, $W_i \to A W_i$ without changing the function), and non-trivial gradient dynamics.

### Polar Decomposition via SVD

For any matrix $W = U \Sigma V^\top$ (SVD), the polar factors are:
- $Q = U V^\top$ (the nearest orthogonal matrix to $W$)
- $P = V \Sigma V^\top$ (symmetric positive semi-definite)

This gives $W = QP$, decomposing each weight into its **orientation** ($Q$, which basis directions it maps to) and **spectrum** ($P$, how much it stretches along each direction).

### Newton-Schulz Orthogonalization (Muon's Core)

Muon replaces the raw gradient $G$ with the closest orthogonal matrix $\hat{G}$, computed via iterative Newton-Schulz: $X_{k+1} = \frac{3}{2}X_k - \frac{1}{2}X_k(X_k^\top X_k)$. This converges cubically to the orthogonal polar factor of $G$, effectively projecting the gradient onto the Stiefel manifold.

In [ ]:
def init_weights(num_layers, seed=42):
    """Initialize layers near identity for stability."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def forward(weights, X):
    """Forward pass: W_L @ ... @ W_1 @ X."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, target):
    """Loss = 0.5 * ||W_product @ X - T @ X||^2 / N."""
    pred = forward(weights, X)
    target_out = target @ X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients(weights, X, target):
    """Backprop through deep linear net."""
    num_layers = len(weights)
    N = X.shape[1]

    # Forward pass storing activations
    activations = [X.copy()]
    out = X.copy()
    for W in weights:
        out = W @ out
        activations.append(out.copy())

    # Backward pass
    target_out = target @ X
    delta = (activations[-1] - target_out) / N

    grads = []
    for i in range(num_layers - 1, -1, -1):
        G = delta @ activations[i].T
        grads.insert(0, G)
        if i > 0:
            delta = weights[i].T @ delta

    return grads

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=NS_ITERS):
    """
    Newton-Schulz iteration to approximate the orthogonal polar factor.
    Returns closest orthogonal matrix to G (i.e., U @ V^T from SVD).
    """
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A

    return X

In [ ]:
def polar_decomposition(W):
    """
    Compute the polar decomposition W = Q P.
    Q is orthogonal (or unitary), P is symmetric positive semi-definite.
    Uses SVD: W = U S V^T => Q = U V^T, P = V S V^T.
    """
    U, S, Vt = np.linalg.svd(W, full_matrices=True)
    Q = U @ Vt
    P = Vt.T @ np.diag(S) @ Vt
    return Q, P

In [ ]:
def find_stable_lr_sgd():
    """Find maximum stable SGD learning rate."""
    candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]
    for lr in candidates:
        np.random.seed(42)
        weights = init_weights(NUM_LAYERS)
        velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
        initial_loss = compute_loss(weights, X_data, W_target)
        stable = True
        for step in range(100):
            grads = compute_gradients(weights, X_data, W_target)
            for i in range(NUM_LAYERS):
                velocities[i] = MOMENTUM * velocities[i] + grads[i]
                weights[i] -= lr * velocities[i]
            loss = compute_loss(weights, X_data, W_target)
            if np.isnan(loss) or loss > initial_loss * 50:
                stable = False
                break
        if stable:
            return lr
    return 0.001

## Optimizer Step Functions

Two optimizers are compared:

1. **SGD with momentum:** $v_t = \beta v_{t-1} + G_t$, $W_t = W_{t-1} - \eta v_t$. The raw gradient $G$ encodes both orientation and spectrum information without any geometric projection.

2. **Muon with momentum:** $\hat{G}_t = \mathrm{NS}(G_t)$, $v_t = \beta v_{t-1} + \hat{G}_t$, $W_t = W_{t-1} - \eta v_t$. By orthogonalizing the gradient before accumulation, Muon's updates have unit singular values -- the update direction is an orthogonal matrix, which is precisely a pure rotation/reflection. The key question is whether this property of the *gradient* translates into preferential movement in the *orientation* component of the weight itself.

In [ ]:
def sgd_step(weights, velocities, lr):
    """One step of SGD with momentum."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        velocities[i] = MOMENTUM * velocities[i] + grads[i]
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities

In [ ]:
def muon_step(weights, velocities, lr):
    """One step of Muon with momentum."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        ortho_grad = newton_schulz_orthogonalize(grads[i])
        velocities[i] = MOMENTUM * velocities[i] + ortho_grad
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities

## Polar Coordinate Tracking Engine

This is the core measurement apparatus. At every optimization step $t$, for each layer $i$:

1. **Compute** $W_t^{(i)} = Q_t^{(i)} P_t^{(i)}$ via SVD-based polar decomposition.
2. **Per-step ratios:** $\|\Delta Q\|_F / \|\Delta W\|_F$ and $\|\Delta P\|_F / \|\Delta W\|_F$ -- what fraction of the weight change is orientation vs. spectrum.
3. **Cumulative drift:** $\|Q_t - Q_0\|_F$ and $\|P_t - P_0\|_F$ -- total displacement from initialization in each subspace.

**Important subtlety:** $\|\Delta Q\|_F + \|\Delta P\|_F \neq \|\Delta W\|_F$ in general (the triangle inequality does not saturate for polar decomposition). The ratios can individually exceed 1. What matters is the *relative* comparison between optimizers, not absolute values.

In [ ]:
def run_and_track_polar(optimizer, lr, num_steps):
    """
    Run optimizer for num_steps, at each step decompose each layer's W_t = Q_t P_t.
    Track per-step and cumulative polar decomposition metrics.

    Returns dict with per-layer and averaged metrics.
    """
    np.random.seed(42)
    weights = init_weights(NUM_LAYERS)
    velocities = [np.zeros_like(w) for w in weights]

    # Initial polar decompositions
    Q_prev = []
    P_prev = []
    Q_init = []
    P_init = []
    for i in range(NUM_LAYERS):
        Q, P = polar_decomposition(weights[i])
        Q_prev.append(Q.copy())
        P_prev.append(P.copy())
        Q_init.append(Q.copy())
        P_init.append(P.copy())

    # Per-layer tracking arrays
    # Per-step ratios: ||DeltaQ||/||DeltaW|| and ||DeltaP||/||DeltaW||
    dQ_ratio = np.zeros((NUM_LAYERS, num_steps))  # ||DeltaQ||/||DeltaW||
    dP_ratio = np.zeros((NUM_LAYERS, num_steps))  # ||DeltaP||/||DeltaW||

    # Per-step absolute norms
    dQ_norm = np.zeros((NUM_LAYERS, num_steps))
    dP_norm = np.zeros((NUM_LAYERS, num_steps))
    dW_norm = np.zeros((NUM_LAYERS, num_steps))

    # Cumulative drift from initialization
    cum_Q_drift = np.zeros((NUM_LAYERS, num_steps))
    cum_P_drift = np.zeros((NUM_LAYERS, num_steps))
    cum_W_drift = np.zeros((NUM_LAYERS, num_steps))

    # Loss tracking
    losses = np.zeros(num_steps + 1)
    losses[0] = compute_loss(weights, X_data, W_target)

    W_prev = [w.copy() for w in weights]
    W_init = [w.copy() for w in weights]

    print(f"    [{optimizer.upper()}] Initial loss: {losses[0]:.6e}")
    print(f"    [{optimizer.upper()}] Tracking arrays allocated: dQ_ratio, dP_ratio, cum_drift -- shapes ({NUM_LAYERS}, {num_steps})")

    for step in range(num_steps):
        # Take optimizer step
        if optimizer == 'sgd':
            weights, velocities = sgd_step(weights, velocities, lr)
        elif optimizer == 'muon':
            weights, velocities = muon_step(weights, velocities, lr)

        losses[step + 1] = compute_loss(weights, X_data, W_target)

        # Check for divergence
        if np.isnan(losses[step + 1]) or losses[step + 1] > 1e10:
            print(f"    WARNING: {optimizer} diverged at step {step + 1}")
            # Fill remaining with NaN
            dQ_ratio[:, step:] = np.nan
            dP_ratio[:, step:] = np.nan
            dQ_norm[:, step:] = np.nan
            dP_norm[:, step:] = np.nan
            dW_norm[:, step:] = np.nan
            cum_Q_drift[:, step:] = np.nan
            cum_P_drift[:, step:] = np.nan
            cum_W_drift[:, step:] = np.nan
            losses[step + 1:] = np.nan
            break

        for i in range(NUM_LAYERS):
            # Polar decomposition of current weight
            Q_curr, P_curr = polar_decomposition(weights[i])

            # Step differences
            delta_W = weights[i] - W_prev[i]
            delta_Q = Q_curr - Q_prev[i]
            delta_P = P_curr - P_prev[i]

            norm_dW = np.linalg.norm(delta_W, 'fro')
            norm_dQ = np.linalg.norm(delta_Q, 'fro')
            norm_dP = np.linalg.norm(delta_P, 'fro')

            dW_norm[i, step] = norm_dW
            dQ_norm[i, step] = norm_dQ
            dP_norm[i, step] = norm_dP

            if norm_dW > 1e-15:
                dQ_ratio[i, step] = norm_dQ / norm_dW
                dP_ratio[i, step] = norm_dP / norm_dW
            else:
                dQ_ratio[i, step] = np.nan
                dP_ratio[i, step] = np.nan

            # Cumulative drift from initialization
            cum_Q_drift[i, step] = np.linalg.norm(Q_curr - Q_init[i], 'fro')
            cum_P_drift[i, step] = np.linalg.norm(P_curr - P_init[i], 'fro')
            cum_W_drift[i, step] = np.linalg.norm(weights[i] - W_init[i], 'fro')

            # Update previous
            Q_prev[i] = Q_curr.copy()
            P_prev[i] = P_curr.copy()

        W_prev = [w.copy() for w in weights]

        # Periodic progress snapshots
        if (step + 1) in [1, 10, 50, 100, 150, 200, 250, 300]:
            avg_dQr = np.nanmean(dQ_ratio[:, step])
            avg_dPr = np.nanmean(dP_ratio[:, step])
            avg_cum_Q = np.nanmean(cum_Q_drift[:, step])
            avg_cum_P = np.nanmean(cum_P_drift[:, step])
            q_frac = avg_cum_Q / (avg_cum_Q + avg_cum_P + 1e-15)
            print(f"    [{optimizer.upper()}] Step {step+1:3d}: loss={losses[step+1]:.4e} | "
                  f"dQ/dW={avg_dQr:.4f} dP/dW={avg_dPr:.4f} | "
                  f"cum_Q={avg_cum_Q:.4f} cum_P={avg_cum_P:.4f} Q_frac={q_frac:.4f}")

    return {
        'dQ_ratio': dQ_ratio,       # (NUM_LAYERS, num_steps)
        'dP_ratio': dP_ratio,
        'dQ_norm': dQ_norm,
        'dP_norm': dP_norm,
        'dW_norm': dW_norm,
        'cum_Q_drift': cum_Q_drift,
        'cum_P_drift': cum_P_drift,
        'cum_W_drift': cum_W_drift,
        'losses': losses,
    }

## Main Experiment: Initialization and Sanity Checks

Before running the optimizers, we verify:
1. The initial loss is finite and reasonable (network near identity, target is random, so loss should be $O(1)$).
2. The polar decomposition is numerically accurate: reconstruction error $\|W - QP\|$ should be near machine epsilon, $Q$ should be orthogonal, $P$ should be symmetric with non-negative eigenvalues.
3. SGD's learning rate is auto-tuned to the maximum stable value, ensuring a fair comparison (Muon can tolerate higher effective LR due to spectral normalization of gradients).

In [ ]:
print("=" * 100)
print("1.2b-ii: TRAJECTORY IN POLAR COORDINATES -- (Q_t, P_t) DECOMPOSITION")
print("=" * 100)
print(f"Setup: {NUM_LAYERS}-layer deep linear net (dim={DIM}), quadratic loss, {NUM_STEPS} steps")
print(f"LR_Muon={LR_MUON}, Momentum={MOMENTUM}")
print(f"Report at steps: {REPORT_STEPS}")
print("=" * 100)

In [ ]:
# Find stable SGD learning rate
lr_sgd = find_stable_lr_sgd()
print(f"\nSGD learning rate (max stable): {lr_sgd}")
print(f"Muon learning rate (fixed):     {LR_MUON}")
print(f"LR ratio (Muon/SGD):            {LR_MUON/lr_sgd:.2f}x")
print(f"\nNote: Muon's gradient orthogonalization bounds the update spectral norm,")
print(f"allowing potentially larger effective learning rates. The LR difference")
print(f"is an inherent property of the optimizers, not a confound.")

In [ ]:
# Quick sanity check
np.random.seed(42)
w_test = init_weights(NUM_LAYERS)
loss_init = compute_loss(w_test, X_data, W_target)
print(f"\nInitial loss: {loss_init:.6e}")

In [ ]:
# Verify polar decomposition works
print("\nVerifying polar decomposition on initial weights...")
for i, W in enumerate(w_test):
    Q, P = polar_decomposition(W)
    recon_err = np.linalg.norm(W - Q @ P, 'fro') / np.linalg.norm(W, 'fro')
    Q_orth_err = np.linalg.norm(Q.T @ Q - np.eye(DIM), 'fro')
    P_sym_err = np.linalg.norm(P - P.T, 'fro')
    P_eigmin = np.min(np.linalg.eigvalsh(P))
    print(f"  Layer {i}: recon_err={recon_err:.2e}, Q_orth_err={Q_orth_err:.2e}, "
          f"P_sym_err={P_sym_err:.2e}, P_min_eig={P_eigmin:.4f}")

## Running Both Optimizers with Polar Tracking

Each optimizer runs for 300 steps from **identical initialization** (same seed). At every step, the full polar decomposition is computed for all 4 layers -- this means $4 \times 300 = 1200$ SVD computations per optimizer.

Watch the periodic snapshots below for:
- **dQ/dW and dP/dW:** Per-step ratios. If Muon's dQ/dW >> dP/dW, it's moving primarily in orientation space.
- **cum_Q and cum_P:** How far each factor has drifted from initialization. Growing cum_Q with stable cum_P means the optimizer is "rotating without stretching."
- **Q_frac:** The fraction $\|Q_t - Q_0\| / (\|Q_t - Q_0\| + \|P_t - P_0\|)$. Values near 1.0 = pure orientation change; 0.5 = equal mix.

In [ ]:
print(f"\n{'=' * 100}")
print("RUNNING OPTIMIZERS AND TRACKING POLAR DECOMPOSITION")
print("=" * 100)

In [ ]:
print("\n  Running SGD...", flush=True)
results_sgd = run_and_track_polar('sgd', lr_sgd, NUM_STEPS)
print(f"    Final loss: {results_sgd['losses'][-1]:.6e}")
print(f"    Loss reduction factor: {results_sgd['losses'][0] / (results_sgd['losses'][-1] + 1e-15):.1f}x")
print(f"    Result arrays: {list(results_sgd.keys())}")
print(f"    dQ_ratio range: [{np.nanmin(results_sgd['dQ_ratio']):.4f}, {np.nanmax(results_sgd['dQ_ratio']):.4f}]")
print(f"    dP_ratio range: [{np.nanmin(results_sgd['dP_ratio']):.4f}, {np.nanmax(results_sgd['dP_ratio']):.4f}]")

In [ ]:
print("\n  Running Muon...", flush=True)
results_muon = run_and_track_polar('muon', LR_MUON, NUM_STEPS)
print(f"    Final loss: {results_muon['losses'][-1]:.6e}")
print(f"    Loss reduction factor: {results_muon['losses'][0] / (results_muon['losses'][-1] + 1e-15):.1f}x")
print(f"    Result arrays: {list(results_muon.keys())}")
print(f"    dQ_ratio range: [{np.nanmin(results_muon['dQ_ratio']):.4f}, {np.nanmax(results_muon['dQ_ratio']):.4f}]")
print(f"    dP_ratio range: [{np.nanmin(results_muon['dP_ratio']):.4f}, {np.nanmax(results_muon['dP_ratio']):.4f}]")

# Immediate comparison
print(f"\n  --- Quick Comparison ---")
print(f"  SGD  overall mean dQ/dW = {np.nanmean(results_sgd['dQ_ratio']):.4f},  dP/dW = {np.nanmean(results_sgd['dP_ratio']):.4f}")
print(f"  Muon overall mean dQ/dW = {np.nanmean(results_muon['dQ_ratio']):.4f},  dP/dW = {np.nanmean(results_muon['dP_ratio']):.4f}")
print(f"  -> Muon dQ/dW is {'higher' if np.nanmean(results_muon['dQ_ratio']) > np.nanmean(results_sgd['dQ_ratio']) else 'lower'} than SGD")
print(f"  -> Muon dP/dW is {'higher' if np.nanmean(results_muon['dP_ratio']) > np.nanmean(results_sgd['dP_ratio']) else 'lower'} than SGD")

## Table 1: Per-Step Orientation vs. Spectrum Change Ratios

This table shows $\|\Delta Q\|/\|\Delta W\|$ and $\|\Delta P\|/\|\Delta W\|$ at selected checkpoints, averaged over a 10-step window and across all 4 layers.

**What to look for:**
- If Muon's $\|\Delta Q\|/\|\Delta W\|$ is consistently higher than SGD's, Muon preferentially rotates weights.
- If Muon's $\|\Delta P\|/\|\Delta W\|$ is consistently lower, Muon preserves the spectrum.
- If both ratios are similar between optimizers, the polar decomposition does not distinguish them.
- Watch for trends: do the ratios change as training progresses (transient vs. steady-state)?

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 1: PER-STEP ORIENTATION vs SPECTRUM CHANGE RATIOS")
print("        (averaged across layers, averaged over 10-step window around report step)")
print("=" * 100)

In [ ]:
print(f"\n{'Step':>6} | {'SGD ||dQ||/||dW||':>18} | {'SGD ||dP||/||dW||':>18} | "
      f"{'Muon ||dQ||/||dW||':>19} | {'Muon ||dP||/||dW||':>19}")
print("-" * 90)

In [ ]:
for step in REPORT_STEPS:
    # Average over a window of 10 steps centered on the report step
    lo = max(0, step - 5)
    hi = min(NUM_STEPS, step + 5)

    sgd_dQ_r = np.nanmean(results_sgd['dQ_ratio'][:, lo:hi])
    sgd_dP_r = np.nanmean(results_sgd['dP_ratio'][:, lo:hi])
    muon_dQ_r = np.nanmean(results_muon['dQ_ratio'][:, lo:hi])
    muon_dP_r = np.nanmean(results_muon['dP_ratio'][:, lo:hi])

    print(f"{step:6d} | {sgd_dQ_r:18.6f} | {sgd_dP_r:18.6f} | "
          f"{muon_dQ_r:19.6f} | {muon_dP_r:19.6f}")

In [ ]:
# Overall averages
sgd_dQ_overall = np.nanmean(results_sgd['dQ_ratio'])
sgd_dP_overall = np.nanmean(results_sgd['dP_ratio'])
muon_dQ_overall = np.nanmean(results_muon['dQ_ratio'])
muon_dP_overall = np.nanmean(results_muon['dP_ratio'])

In [ ]:
print("-" * 90)
print(f"{'ALL':>6} | {sgd_dQ_overall:18.6f} | {sgd_dP_overall:18.6f} | "
      f"{muon_dQ_overall:19.6f} | {muon_dP_overall:19.6f}")

In [ ]:
# Scientific interpretation of Table 1
print("\n  --- Interpretation of Table 1 ---")
muon_q_dominance = np.nanmean(results_muon['dQ_ratio']) / (np.nanmean(results_muon['dP_ratio']) + 1e-15)
sgd_q_dominance = np.nanmean(results_sgd['dQ_ratio']) / (np.nanmean(results_sgd['dP_ratio']) + 1e-15)
print(f"  Muon dQ/dP dominance ratio: {muon_q_dominance:.4f} (>1 means orientation-dominated)")
print(f"  SGD  dQ/dP dominance ratio: {sgd_q_dominance:.4f}")
if muon_q_dominance > sgd_q_dominance * 1.1:
    print(f"  -> Muon shows stronger orientation dominance ({muon_q_dominance/sgd_q_dominance:.2f}x more than SGD)")
elif sgd_q_dominance > muon_q_dominance * 1.1:
    print(f"  -> SGD shows stronger orientation dominance -- CONTRARY to hypothesis")
else:
    print(f"  -> Both optimizers have similar dQ/dP profiles -- polar decomposition may not distinguish them")

## Table 2: Cumulative Drift from Initialization

While Table 1 showed per-step instantaneous ratios, this table shows the **total displacement** in each polar factor from the initial weights. This captures the cumulative effect: even if per-step ratios are similar, one optimizer might consistently drift more in one direction.

**Key metrics:**
- $\|Q_t - Q_0\|_F$: Total orientation drift. For orthogonal matrices in $\mathbb{R}^{n \times n}$, the maximum possible distance is $2\sqrt{n}$ (from $I$ to $-I$).
- $\|P_t - P_0\|_F$: Total spectrum drift. Unbounded in principle but constrained by training stability.
- $\|W_t - W_0\|_F$: Total weight drift for reference.

**What to watch:** If Muon's Q-drift grows while its P-drift stays small relative to SGD, Muon is "orbiting" on the Stiefel manifold.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 2: CUMULATIVE DRIFT FROM INITIALIZATION (averaged across layers)")
print("=" * 100)

In [ ]:
print(f"\n{'Step':>6} | {'SGD ||Q-Q0||':>14} | {'SGD ||P-P0||':>14} | {'SGD ||W-W0||':>14} | "
      f"{'Muon ||Q-Q0||':>14} | {'Muon ||P-P0||':>14} | {'Muon ||W-W0||':>14}")
print("-" * 105)

In [ ]:
for step in REPORT_STEPS:
    idx = step - 1  # 0-indexed
    sgd_Qd = np.nanmean(results_sgd['cum_Q_drift'][:, idx])
    sgd_Pd = np.nanmean(results_sgd['cum_P_drift'][:, idx])
    sgd_Wd = np.nanmean(results_sgd['cum_W_drift'][:, idx])
    muon_Qd = np.nanmean(results_muon['cum_Q_drift'][:, idx])
    muon_Pd = np.nanmean(results_muon['cum_P_drift'][:, idx])
    muon_Wd = np.nanmean(results_muon['cum_W_drift'][:, idx])

    print(f"{step:6d} | {sgd_Qd:14.6f} | {sgd_Pd:14.6f} | {sgd_Wd:14.6f} | "
          f"{muon_Qd:14.6f} | {muon_Pd:14.6f} | {muon_Wd:14.6f}")

## Table 3: Orientation Fraction -- The Key Diagnostic

This is the most direct test of the hypothesis. The **Q/(Q+P) fraction** normalizes the cumulative drifts into a single number between 0 and 1:

- $Q/(Q+P) \approx 1.0$: Movement is almost entirely orientation (gauge) changes -- consistent with Muon as gauge fixer.
- $Q/(Q+P) \approx 0.5$: Equal orientation and spectrum drift -- no geometric preference.
- $Q/(Q+P) \approx 0.0$: Movement is almost entirely spectrum changes.

The **Q/P ratio** gives the same information on an unbounded scale (values > 1 mean orientation-dominated).

**Critical comparison:** If Muon's Q/(Q+P) significantly exceeds SGD's at all time points, this is strong evidence for the gauge-fixing hypothesis. If they're similar, we need a different geometric lens.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 3: RATIO OF CUMULATIVE Q-DRIFT TO P-DRIFT (averaged across layers)")
print("         Q-drift / P-drift > 1 means more orientation change than spectrum change")
print("=" * 100)

In [ ]:
print(f"\n{'Step':>6} | {'SGD Q/P ratio':>14} | {'Muon Q/P ratio':>15} | {'SGD Q/(Q+P)':>12} | {'Muon Q/(Q+P)':>13}")
print("-" * 72)

In [ ]:
for step in REPORT_STEPS:
    idx = step - 1
    sgd_Qd = np.nanmean(results_sgd['cum_Q_drift'][:, idx])
    sgd_Pd = np.nanmean(results_sgd['cum_P_drift'][:, idx])
    muon_Qd = np.nanmean(results_muon['cum_Q_drift'][:, idx])
    muon_Pd = np.nanmean(results_muon['cum_P_drift'][:, idx])

    sgd_qp = sgd_Qd / sgd_Pd if sgd_Pd > 1e-15 else np.inf
    muon_qp = muon_Qd / muon_Pd if muon_Pd > 1e-15 else np.inf
    sgd_frac = sgd_Qd / (sgd_Qd + sgd_Pd) if (sgd_Qd + sgd_Pd) > 1e-15 else np.nan
    muon_frac = muon_Qd / (muon_Qd + muon_Pd) if (muon_Qd + muon_Pd) > 1e-15 else np.nan

    print(f"{step:6d} | {sgd_qp:14.4f} | {muon_qp:15.4f} | {sgd_frac:12.4f} | {muon_frac:13.4f}")

## Table 4: Per-Layer Breakdown at Step 300

The layer-averaged results may hide important layer-dependent structure. In deep linear networks, the gauge symmetry acts between **adjacent layers** -- so the first and last layers (which interface with input/output) may behave differently from interior layers.

**What to look for:**
- Are the ratios consistent across layers, or does one layer dominate?
- Do interior layers (1, 2) show different Q/P profiles than boundary layers (0, 3)?
- Is the layer-to-layer variance large (suggesting the average is misleading)?

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 4: PER-LAYER BREAKDOWN AT STEP 300")
print("=" * 100)

In [ ]:
print(f"\n  SGD (lr={lr_sgd}):")
print(f"  {'Layer':>6} | {'||dQ||/||dW||':>14} | {'||dP||/||dW||':>14} | "
      f"{'cum ||Q-Q0||':>14} | {'cum ||P-P0||':>14} | {'Q/(Q+P)':>10}")
print("  " + "-" * 85)

In [ ]:
for layer in range(NUM_LAYERS):
    dQr = np.nanmean(results_sgd['dQ_ratio'][layer, :])
    dPr = np.nanmean(results_sgd['dP_ratio'][layer, :])
    Qd = results_sgd['cum_Q_drift'][layer, -1]
    Pd = results_sgd['cum_P_drift'][layer, -1]
    frac = Qd / (Qd + Pd) if (Qd + Pd) > 1e-15 else np.nan
    print(f"  {layer:6d} | {dQr:14.6f} | {dPr:14.6f} | {Qd:14.6f} | {Pd:14.6f} | {frac:10.4f}")

In [ ]:
print(f"\n  Muon (lr={LR_MUON}):")
print(f"  {'Layer':>6} | {'||dQ||/||dW||':>14} | {'||dP||/||dW||':>14} | "
      f"{'cum ||Q-Q0||':>14} | {'cum ||P-P0||':>14} | {'Q/(Q+P)':>10}")
print("  " + "-" * 85)

In [ ]:
for layer in range(NUM_LAYERS):
    dQr = np.nanmean(results_muon['dQ_ratio'][layer, :])
    dPr = np.nanmean(results_muon['dP_ratio'][layer, :])
    Qd = results_muon['cum_Q_drift'][layer, -1]
    Pd = results_muon['cum_P_drift'][layer, -1]
    frac = Qd / (Qd + Pd) if (Qd + Pd) > 1e-15 else np.nan
    print(f"  {layer:6d} | {dQr:14.6f} | {dPr:14.6f} | {Qd:14.6f} | {Pd:14.6f} | {frac:10.4f}")

In [ ]:
# Quantify inter-layer variability
print("\n  --- Layer Variability Analysis ---")
for opt_name, results in [('SGD', results_sgd), ('Muon', results_muon)]:
    q_fracs = []
    for layer in range(NUM_LAYERS):
        Qd = results['cum_Q_drift'][layer, -1]
        Pd = results['cum_P_drift'][layer, -1]
        q_fracs.append(Qd / (Qd + Pd) if (Qd + Pd) > 1e-15 else np.nan)
    q_fracs = np.array(q_fracs)
    print(f"  {opt_name}: Q/(Q+P) per layer = {[f'{x:.4f}' for x in q_fracs]}")
    print(f"    mean={np.nanmean(q_fracs):.4f}, std={np.nanstd(q_fracs):.4f}, "
          f"range=[{np.nanmin(q_fracs):.4f}, {np.nanmax(q_fracs):.4f}]")
    if np.nanstd(q_fracs) > 0.05:
        print(f"    -> Significant layer-to-layer variation (std > 0.05)")
    else:
        print(f"    -> Consistent across layers (std <= 0.05)")

## Table 5: Early vs. Late Training Dynamics

Training dynamics can change qualitatively between phases:

- **Early training (steps 1-50):** Large gradients, rapid loss decrease, weights far from any minimum. The optimizer's geometric bias may be most visible here, or it may be overwhelmed by the strong gradient signal.
- **Late training (steps 250-300):** Small gradients, near convergence, fine-tuning regime. Geometric biases may become more or less pronounced.

**Key question:** Does the orientation/spectrum balance change over training? If Muon starts with balanced Q/P updates but transitions to Q-dominated updates late (or vice versa), this reveals time-dependent geometric behavior that a single summary statistic would miss.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 5: EARLY (steps 1-50) vs LATE (steps 250-300) TRAINING DYNAMICS")
print("=" * 100)

In [ ]:
early_slice = slice(0, 50)
late_slice = slice(250, 300)

In [ ]:
print(f"\n{'Phase':>10} | {'SGD dQ/dW':>12} | {'SGD dP/dW':>12} | "
      f"{'Muon dQ/dW':>12} | {'Muon dP/dW':>12}")
print("-" * 68)

In [ ]:
for phase_name, sl in [('Early', early_slice), ('Late', late_slice)]:
    sgd_dQr = np.nanmean(results_sgd['dQ_ratio'][:, sl])
    sgd_dPr = np.nanmean(results_sgd['dP_ratio'][:, sl])
    muon_dQr = np.nanmean(results_muon['dQ_ratio'][:, sl])
    muon_dPr = np.nanmean(results_muon['dP_ratio'][:, sl])
    print(f"{phase_name:>10} | {sgd_dQr:12.6f} | {sgd_dPr:12.6f} | "
          f"{muon_dQr:12.6f} | {muon_dPr:12.6f}")

In [ ]:
# Quantify the early-to-late transition
print("\n  --- Early-to-Late Transition Analysis ---")
for opt_name, results in [('SGD', results_sgd), ('Muon', results_muon)]:
    early_dQr = np.nanmean(results['dQ_ratio'][:, early_slice])
    late_dQr = np.nanmean(results['dQ_ratio'][:, late_slice])
    early_dPr = np.nanmean(results['dP_ratio'][:, early_slice])
    late_dPr = np.nanmean(results['dP_ratio'][:, late_slice])
    early_qdom = early_dQr / (early_dPr + 1e-15)
    late_qdom = late_dQr / (late_dPr + 1e-15)
    print(f"  {opt_name}: Q-dominance early={early_qdom:.4f}, late={late_qdom:.4f}, "
          f"change={late_qdom - early_qdom:+.4f}")
    if late_qdom > early_qdom * 1.2:
        print(f"    -> {opt_name} becomes MORE orientation-dominated over training")
    elif early_qdom > late_qdom * 1.2:
        print(f"    -> {opt_name} becomes LESS orientation-dominated over training")
    else:
        print(f"    -> {opt_name}'s orientation/spectrum balance is stable across training")

## Plot 1: Polar Decomposition Dynamics (6-Panel Overview)

This figure contains six panels that together paint the full picture of Q/P dynamics:

- **(a) Cumulative Orientation Drift** $\|Q_t - Q_0\|_F$: How far Q has moved from initialization. If Muon (red) rises faster than SGD (blue), Muon changes orientation more aggressively.
- **(b) Cumulative Spectrum Drift** $\|P_t - P_0\|_F$: How far P has moved. If Muon stays lower than SGD, Muon preserves the spectrum.
- **(c) Orientation Fraction** $Q/(Q+P)$: The normalized diagnostic. Values above 0.5 mean orientation-dominated movement; the gray dashed line marks equal Q/P.
- **(d) Per-Step** $\|\Delta Q\|/\|\Delta W\|$: Instantaneous orientation change fraction (smoothed). Look for systematic separation between optimizers.
- **(e) Per-Step** $\|\Delta P\|/\|\Delta W\|$: Instantaneous spectrum change fraction (smoothed). Mirror image of (d).
- **(f) Training Loss**: Reference curve to correlate geometric behavior with optimization progress.

**What to look for:** Panel (c) is the most diagnostic -- a clear and sustained gap between Muon and SGD would strongly support the gauge-fixing hypothesis.

In [ ]:
print(f"\n\n{'=' * 100}")
print("GENERATING PLOTS")
print("=" * 100)

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('1.2b-ii: Trajectory in Polar Coordinates -- (Q, P) Decomposition\n'
                 f'{NUM_LAYERS}-layer linear net, dim={DIM}, {NUM_STEPS} steps',
                 fontsize=14, fontweight='bold')

    t_axis = np.arange(NUM_STEPS)

    # --- Panel (a): Cumulative Q drift ---
    ax = axes[0, 0]
    ax.set_title('(a) Cumulative Orientation Drift ||Q_t - Q_0||')
    for layer in range(NUM_LAYERS):
        ax.plot(t_axis, results_sgd['cum_Q_drift'][layer, :],
                'b-', alpha=0.3, linewidth=0.8)
        ax.plot(t_axis, results_muon['cum_Q_drift'][layer, :],
                'r-', alpha=0.3, linewidth=0.8)
    # Layer-averaged
    ax.plot(t_axis, np.mean(results_sgd['cum_Q_drift'], axis=0),
            'b-', linewidth=2.5, label='SGD (avg)')
    ax.plot(t_axis, np.mean(results_muon['cum_Q_drift'], axis=0),
            'r-', linewidth=2.5, label='Muon (avg)')
    ax.set_xlabel('Step')
    ax.set_ylabel('||Q_t - Q_0||_F')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Panel (b): Cumulative P drift ---
    ax = axes[0, 1]
    ax.set_title('(b) Cumulative Spectrum Drift ||P_t - P_0||')
    for layer in range(NUM_LAYERS):
        ax.plot(t_axis, results_sgd['cum_P_drift'][layer, :],
                'b-', alpha=0.3, linewidth=0.8)
        ax.plot(t_axis, results_muon['cum_P_drift'][layer, :],
                'r-', alpha=0.3, linewidth=0.8)
    ax.plot(t_axis, np.mean(results_sgd['cum_P_drift'], axis=0),
            'b-', linewidth=2.5, label='SGD (avg)')
    ax.plot(t_axis, np.mean(results_muon['cum_P_drift'], axis=0),
            'r-', linewidth=2.5, label='Muon (avg)')
    ax.set_xlabel('Step')
    ax.set_ylabel('||P_t - P_0||_F')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Panel (c): Q drift / (Q drift + P drift) over time ---
    ax = axes[0, 2]
    ax.set_title('(c) Orientation Fraction: ||Q-Q0|| / (||Q-Q0|| + ||P-P0||)')
    sgd_Q_avg = np.mean(results_sgd['cum_Q_drift'], axis=0)
    sgd_P_avg = np.mean(results_sgd['cum_P_drift'], axis=0)
    muon_Q_avg = np.mean(results_muon['cum_Q_drift'], axis=0)
    muon_P_avg = np.mean(results_muon['cum_P_drift'], axis=0)

    sgd_frac = sgd_Q_avg / (sgd_Q_avg + sgd_P_avg + 1e-15)
    muon_frac = muon_Q_avg / (muon_Q_avg + muon_P_avg + 1e-15)

    ax.plot(t_axis, sgd_frac, 'b-', linewidth=2.5, label='SGD')
    ax.plot(t_axis, muon_frac, 'r-', linewidth=2.5, label='Muon')
    ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7,
               label='Equal Q/P')
    ax.set_xlabel('Step')
    ax.set_ylabel('Q fraction')
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Panel (d): Per-step ||dQ||/||dW|| ---
    ax = axes[1, 0]
    ax.set_title('(d) Per-Step ||dQ|| / ||dW|| (layer-averaged)')

    # Smooth with rolling window
    window = 10
    sgd_dQr_avg = np.nanmean(results_sgd['dQ_ratio'], axis=0)
    muon_dQr_avg = np.nanmean(results_muon['dQ_ratio'], axis=0)

    # Rolling mean
    sgd_dQr_smooth = np.convolve(sgd_dQr_avg, np.ones(window)/window, mode='valid')
    muon_dQr_smooth = np.convolve(muon_dQr_avg, np.ones(window)/window, mode='valid')

    ax.plot(np.arange(len(sgd_dQr_smooth)), sgd_dQr_smooth,
            'b-', linewidth=2, label='SGD')
    ax.plot(np.arange(len(muon_dQr_smooth)), muon_dQr_smooth,
            'r-', linewidth=2, label='Muon')
    ax.set_xlabel('Step')
    ax.set_ylabel('||dQ|| / ||dW||')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Panel (e): Per-step ||dP||/||dW|| ---
    ax = axes[1, 1]
    ax.set_title('(e) Per-Step ||dP|| / ||dW|| (layer-averaged)')

    sgd_dPr_avg = np.nanmean(results_sgd['dP_ratio'], axis=0)
    muon_dPr_avg = np.nanmean(results_muon['dP_ratio'], axis=0)

    sgd_dPr_smooth = np.convolve(sgd_dPr_avg, np.ones(window)/window, mode='valid')
    muon_dPr_smooth = np.convolve(muon_dPr_avg, np.ones(window)/window, mode='valid')

    ax.plot(np.arange(len(sgd_dPr_smooth)), sgd_dPr_smooth,
            'b-', linewidth=2, label='SGD')
    ax.plot(np.arange(len(muon_dPr_smooth)), muon_dPr_smooth,
            'r-', linewidth=2, label='Muon')
    ax.set_xlabel('Step')
    ax.set_ylabel('||dP|| / ||dW||')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Panel (f): Loss curves ---
    ax = axes[1, 2]
    ax.set_title('(f) Training Loss')
    ax.semilogy(np.arange(NUM_STEPS + 1), results_sgd['losses'], 'b-',
                linewidth=2, label=f'SGD (lr={lr_sgd})')
    ax.semilogy(np.arange(NUM_STEPS + 1), results_muon['losses'], 'r-',
                linewidth=2, label=f'Muon (lr={LR_MUON})')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(SCRIPT_DIR, 'polar_trajectory_decomposition.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n  Plot saved to: {plot_path}")

In [ ]:
except ImportError:
    print("\n  WARNING: matplotlib not available, skipping plots.")
    plot_path = None

## Plot 2: Phase Portrait -- Q Drift vs. P Drift

The phase portrait plots cumulative Q-drift ($y$-axis) against cumulative P-drift ($x$-axis), creating a trajectory in the "orientation-spectrum" plane. This is a powerful visualization because:

- **Trajectories above the diagonal** ($Q = P$ line) indicate orientation-dominated movement.
- **Trajectories below the diagonal** indicate spectrum-dominated movement.
- The **slope** of the trajectory at any point gives the instantaneous ratio $dQ/dP$.
- Different slopes between optimizers directly show different geometric biases.

**Panel (a)** shows all 4 layers individually (thin lines) with endpoints marked (x). **Panel (b)** shows the layer-averaged trajectory (thick lines). If Muon's trajectory curves upward relative to SGD's, Muon preferentially explores the orientation direction in weight space.

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('1.2b-ii: Phase Portrait -- Cumulative Q Drift vs P Drift',
                 fontsize=13, fontweight='bold')

    # Panel (a): All layers
    ax = axes[0]
    ax.set_title('(a) Per-Layer Phase Portrait')
    for layer in range(NUM_LAYERS):
        ax.plot(results_sgd['cum_P_drift'][layer, :],
                results_sgd['cum_Q_drift'][layer, :],
                'b-', alpha=0.5, linewidth=1, label=f'SGD L{layer}' if layer == 0 else None)
        ax.plot(results_muon['cum_P_drift'][layer, :],
                results_muon['cum_Q_drift'][layer, :],
                'r-', alpha=0.5, linewidth=1, label=f'Muon L{layer}' if layer == 0 else None)
        # Mark endpoints
        ax.plot(results_sgd['cum_P_drift'][layer, -1],
                results_sgd['cum_Q_drift'][layer, -1],
                'bx', markersize=8)
        ax.plot(results_muon['cum_P_drift'][layer, -1],
                results_muon['cum_Q_drift'][layer, -1],
                'rx', markersize=8)

    # Diagonal reference line
    max_val = max(
        np.nanmax(results_sgd['cum_Q_drift']),
        np.nanmax(results_sgd['cum_P_drift']),
        np.nanmax(results_muon['cum_Q_drift']),
        np.nanmax(results_muon['cum_P_drift']),
    )
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.4, label='Q=P (equal)')
    ax.set_xlabel('Cumulative ||P_t - P_0||_F (spectrum drift)')
    ax.set_ylabel('Cumulative ||Q_t - Q_0||_F (orientation drift)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel (b): Averaged
    ax = axes[1]
    ax.set_title('(b) Layer-Averaged Phase Portrait')
    sgd_Q_avg = np.mean(results_sgd['cum_Q_drift'], axis=0)
    sgd_P_avg = np.mean(results_sgd['cum_P_drift'], axis=0)
    muon_Q_avg = np.mean(results_muon['cum_Q_drift'], axis=0)
    muon_P_avg = np.mean(results_muon['cum_P_drift'], axis=0)

    ax.plot(sgd_P_avg, sgd_Q_avg, 'b-', linewidth=2.5, label='SGD')
    ax.plot(muon_P_avg, muon_Q_avg, 'r-', linewidth=2.5, label='Muon')
    ax.plot(sgd_P_avg[-1], sgd_Q_avg[-1], 'bx', markersize=12, markeredgewidth=3)
    ax.plot(muon_P_avg[-1], muon_Q_avg[-1], 'rx', markersize=12, markeredgewidth=3)
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.4, label='Q=P (equal)')
    ax.set_xlabel('Cumulative ||P_t - P_0||_F (spectrum drift)')
    ax.set_ylabel('Cumulative ||Q_t - Q_0||_F (orientation drift)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path2 = os.path.join(SCRIPT_DIR, 'polar_phase_portrait.png')
    plt.savefig(plot_path2, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Plot saved to: {plot_path2}")

In [ ]:
except ImportError:
    pass

## Final Analysis: Summary Statistics

All key metrics are consolidated below. The summary statistics average across all layers and all training steps to give the "big picture" view. These numbers are the primary evidence for or against the hypothesis.

Note that the per-step ratios (dQ/dW, dP/dW) capture the instantaneous geometry, while the cumulative drift and Q/(Q+P) fraction capture the integrated trajectory. Both views are important -- an optimizer could have similar per-step ratios but accumulate differently due to directional consistency.

In [ ]:
print(f"\n\n{'=' * 100}")
print("FINAL ANALYSIS: POLAR COORDINATE TRAJECTORY DECOMPOSITION")
print("=" * 100)

In [ ]:
# Compute key summary statistics
sgd_dQr_all = np.nanmean(results_sgd['dQ_ratio'])
sgd_dPr_all = np.nanmean(results_sgd['dP_ratio'])
muon_dQr_all = np.nanmean(results_muon['dQ_ratio'])
muon_dPr_all = np.nanmean(results_muon['dP_ratio'])

In [ ]:
sgd_cumQ_final = np.nanmean(results_sgd['cum_Q_drift'][:, -1])
sgd_cumP_final = np.nanmean(results_sgd['cum_P_drift'][:, -1])
muon_cumQ_final = np.nanmean(results_muon['cum_Q_drift'][:, -1])
muon_cumP_final = np.nanmean(results_muon['cum_P_drift'][:, -1])

In [ ]:
sgd_Q_frac_final = sgd_cumQ_final / (sgd_cumQ_final + sgd_cumP_final + 1e-15)
muon_Q_frac_final = muon_cumQ_final / (muon_cumQ_final + muon_cumP_final + 1e-15)

In [ ]:
print(f"""
  SUMMARY STATISTICS (averaged across layers, over all {NUM_STEPS} steps):
  ---------------------------------------------------------------
  PER-STEP RATIOS:
    SGD:   ||dQ||/||dW|| = {sgd_dQr_all:.6f},  ||dP||/||dW|| = {sgd_dPr_all:.6f}
    Muon:  ||dQ||/||dW|| = {muon_dQr_all:.6f},  ||dP||/||dW|| = {muon_dPr_all:.6f}

  CUMULATIVE DRIFT AT STEP {NUM_STEPS}:
    SGD:   ||Q-Q0|| = {sgd_cumQ_final:.6f},  ||P-P0|| = {sgd_cumP_final:.6f},  Q/(Q+P) = {sgd_Q_frac_final:.4f}
    Muon:  ||Q-Q0|| = {muon_cumQ_final:.6f},  ||P-P0|| = {muon_cumP_final:.6f},  Q/(Q+P) = {muon_Q_frac_final:.4f}

  FINAL LOSSES:
    SGD:   {results_sgd['losses'][-1]:.6e}
    Muon:  {results_muon['losses'][-1]:.6e}
  ---------------------------------------------------------------
""")

## Hypothesis Testing

Five specific, falsifiable tests derived from the gauge-fixing hypothesis. Each compares a quantitative metric between Muon and SGD:

- **T1:** Muon's per-step $\|\Delta Q\|/\|\Delta W\|$ exceeds SGD's (Muon rotates more per step).
- **T2:** Muon's per-step $\|\Delta P\|/\|\Delta W\|$ is less than SGD's (Muon stretches less per step).
- **T3:** Muon's cumulative Q/(Q+P) fraction exceeds SGD's (total trajectory is more orientation-dominated).
- **T4:** Muon's cumulative Q/P ratio exceeds SGD's (same as T3 on unbounded scale).
- **T5:** Muon's total spectrum drift $\|P - P_0\|$ is less than SGD's (Muon preserves the spectrum better).

**Scoring:**
- 4-5 tests passed = STRONG SUPPORT for gauge-fixing interpretation
- 3 tests = MODERATE SUPPORT
- 2 tests = WEAK SIGNAL
- 0-1 tests = NO SUPPORT (requires reinterpretation)

A special note is added if the two optimizers have very similar Q/(Q+P) profiles (difference < 0.05), suggesting the polar decomposition may not be the right geometric lens for distinguishing them.

In [ ]:
print("  HYPOTHESIS TESTS:")
print("  ---------------------------------------------------------------")

In [ ]:
# Test 1: Muon's per-step dQ/dW > SGD's per-step dQ/dW
# (Muon changes orientation more per step)
test1 = muon_dQr_all > sgd_dQr_all
print(f"  T1: Muon ||dQ||/||dW|| > SGD ||dQ||/||dW||")
print(f"      Muon={muon_dQr_all:.6f} vs SGD={sgd_dQr_all:.6f}")
print(f"      -> {'YES' if test1 else 'NO'}")

In [ ]:
# Test 2: Muon's per-step dP/dW < SGD's per-step dP/dW
# (Muon changes spectrum less per step)
test2 = muon_dPr_all < sgd_dPr_all
print(f"\n  T2: Muon ||dP||/||dW|| < SGD ||dP||/||dW||")
print(f"      Muon={muon_dPr_all:.6f} vs SGD={sgd_dPr_all:.6f}")
print(f"      -> {'YES' if test2 else 'NO'}")

In [ ]:
# Test 3: Muon's cumulative Q fraction > SGD's cumulative Q fraction
# (Muon's total movement is more orientation-dominated)
test3 = muon_Q_frac_final > sgd_Q_frac_final
print(f"\n  T3: Muon Q/(Q+P) > SGD Q/(Q+P) at step {NUM_STEPS}")
print(f"      Muon={muon_Q_frac_final:.4f} vs SGD={sgd_Q_frac_final:.4f}")
print(f"      -> {'YES' if test3 else 'NO'}")

In [ ]:
# Test 4: Muon has higher Q/P ratio overall
muon_QP_ratio = muon_cumQ_final / (muon_cumP_final + 1e-15)
sgd_QP_ratio = sgd_cumQ_final / (sgd_cumP_final + 1e-15)
test4 = muon_QP_ratio > sgd_QP_ratio
print(f"\n  T4: Muon cumulative Q/P ratio > SGD cumulative Q/P ratio")
print(f"      Muon={muon_QP_ratio:.4f} vs SGD={sgd_QP_ratio:.4f}")
print(f"      -> {'YES' if test4 else 'NO'}")

In [ ]:
# Test 5: Muon's cumulative P drift is smaller than SGD's
# (Muon moves less in spectrum space)
test5 = muon_cumP_final < sgd_cumP_final
print(f"\n  T5: Muon ||P-P0|| < SGD ||P-P0|| at step {NUM_STEPS}")
print(f"      Muon={muon_cumP_final:.6f} vs SGD={sgd_cumP_final:.6f}")
print(f"      -> {'YES' if test5 else 'NO'}")

In [ ]:
tests_passed = sum([test1, test2, test3, test4, test5])

In [ ]:
print(f"\n  ---------------------------------------------------------------")
print(f"  Tests passed: {tests_passed}/5")

In [ ]:
# Determine overall verdict
if tests_passed >= 4:
    overall = "STRONG SUPPORT"
    detail = (
        "The data strongly supports the hypothesis that Muon's updates are\n"
        "  predominantly orientation (Q) changes while SGD changes both Q and P.\n"
        "  Muon moves on the Stiefel manifold; SGD wanders in full weight space."
    )
elif tests_passed >= 3:
    overall = "MODERATE SUPPORT"
    detail = (
        "The data moderately supports the polar decomposition hypothesis.\n"
        "  There is a measurable difference in how Muon vs SGD distribute\n"
        "  their updates between orientation (Q) and spectrum (P)."
    )
elif tests_passed >= 2:
    overall = "WEAK SIGNAL"
    detail = (
        "The data shows a weak signal. The difference between Muon and SGD\n"
        "  in polar coordinates is present but not dramatic.\n"
        "  Consistent with 1.2b-i: the distinction may be subtler than expected."
    )
else:
    overall = "NO SUPPORT / SURPRISING"
    detail = (
        "The hypothesis is not supported. Muon does NOT preferentially change\n"
        "  orientation over spectrum, or does so LESS than SGD.\n"
        "  This is a genuine surprise that requires reinterpretation."
    )

In [ ]:
# Check for the SURPRISE case: if both optimizers have very similar profiles
if abs(muon_Q_frac_final - sgd_Q_frac_final) < 0.05:
    overall += " (SIMILAR PROFILES)"
    detail += (
        "\n\n  NOTE: Both optimizers have very similar Q/(Q+P) fractions.\n"
        "  The polar decomposition may not be the right lens to distinguish them.\n"
        "  This is consistent with 1.2b-i's finding that the Lyapunov exponents\n"
        "  were similar for gauge and physical directions."
    )

In [ ]:
print(f"""
  ======================================================================
  VERDICT: {overall}
  ======================================================================
  {detail}
  ======================================================================
""")

In [ ]:
print("=" * 100)
print(f"  Tests passed: {tests_passed}/5")
print(f"  Overall: {overall}")
print("=" * 100)

## Conclusions

### What This Experiment Measures

This experiment performed a complete polar decomposition $W_t = Q_t P_t$ at every step of training for both Muon and SGD+momentum, tracking how each optimizer distributes its weight updates between orientation changes ($Q$, the gauge/rotational degree of freedom) and spectrum changes ($P$, the physically meaningful singular value structure).

### Key Findings

The five hypothesis tests above provide a quantitative scorecard for the gauge-fixing interpretation of Muon. The results should be interpreted in conjunction with:

1. **Experiment 1.2b-i (Lyapunov analysis):** Found that Muon is MORE chaotic in weight space but more stable in loss space. If polar decomposition shows Muon is orientation-dominated, this explains the paradox: chaotic orientation changes (gauge rotations) do not affect the loss, while stable spectrum preservation maintains function quality.

2. **The LR confound:** Muon and SGD operate at different learning rates. Some of the difference in polar dynamics may be attributable to LR rather than geometric structure. Future experiments should test with matched effective step sizes.

3. **Linearity limitation:** Deep linear networks have exact gauge symmetries (orthogonal inter-layer transformations). In nonlinear networks, these symmetries are broken, and the Q/P decomposition may behave differently. Generalization should not be assumed.

### Implications for the Muon-as-Gauge-Fixer Theory

- **If STRONG SUPPORT:** The polar decomposition provides direct evidence that Muon's Newton-Schulz orthogonalization of gradients translates into preferential movement on the Stiefel manifold in weight space. This is consistent with Muon acting as an RG gauge fixer.
- **If WEAK/NO SUPPORT:** The gauge-fixing mechanism, if it exists, may operate in a different geometric subspace than the simple polar decomposition captures. The fiber bundle structure of the gauge symmetry may require a more sophisticated decomposition (e.g., aligned with the specific gauge orbits rather than the generic polar factorization).

### Next Steps

- Compare with Experiment 1.3 (Hessian analysis) to see if orientation vs. spectrum changes correlate with null-space vs. non-null-space directions of the Hessian.
- Test with matched effective step sizes to rule out LR confounds.
- Extend to nonlinear networks to assess generalization.